# End-to-End RAG Pipeline

Construct an enterprise End-to-End Retrieval-Augmented Generation (RAG) pipeline utilizing semantic document ingestion, embedding models, and a Vector Database.

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

C:\Users\Kavya\AppData\Local\Temp\ipykernel_24656\1067146095.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


## Setup

In [2]:
loader = TextLoader("enterprise_data.txt")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = text_splitter.split_documents(documents)

## Document Ingestion and Chunking

In [3]:
with open("enterprise_data.txt", "w") as f:
    f.write(
        "Project Titan was launched in Q3 2025 to overhaul the cloud infrastructure using decentralized nodes. "
    )
    f.write(
        "The initiative successfully reduced server latency by 35% across European data centers. "
    )
    f.write(
        "Sarah Jenkins, the Chief Architect, authored the final deployment protocols in December 2025."
    )

## Vector Database and Embeddings

In [4]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")

vector_db = Chroma.from_documents(
    documents=chunks, embedding=embeddings, persist_directory="./chroma_storage"
)
retriever = vector_db.as_retriever(search_kwargs={"k": 2})

## RAG Pipeline Construction

In [5]:
llm = ChatOllama(model="lfm3:1.7b", temperature=0)

template = """Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know.

Context: {context}

Question: {question}

Answer:"""
prompt = PromptTemplate.from_template(template)


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## Execution and Querying

In [6]:
query = "What impact did Project Titan have and who was the Chief Architect?"

retrieved_docs = retriever.invoke(query)
print("=== Retrieved Semantic Chunks ===")
for i, doc in enumerate(retrieved_docs):
    print(f"Chunk {i+1}: {doc.page_content}")

print("\n=== Final RAG Output ===")
response = rag_chain.invoke(query)
print(response)

=== Retrieved Semantic Chunks ===
Chunk 1: Project Titan was launched in Q3 2025 to overhaul the cloud infrastructure using decentralized
Chunk 2: by 35% across European data centers. Sarah Jenkins, the Chief Architect, authored the final

=== Final RAG Output ===
Project Titan’s launch in Q3 2025 was intended to overhaul the cloud infrastructure—specifically by making changes “using decentralized” approaches and achieving a 35% shift across European data centers. The chief architect responsible for authoring the final plan was **Sarah Jenkins**.
